# voice-pipes — benchmark launcher

Drives the orchestrator HTTP API to **start** benchmark experiments.
Result analysis lives in the admin UI; this notebook only:

1. Health-checks the orchestrator
2. Builds / picks a suite
3. Creates an experiment
4. Kicks off a run (with optional speed / timeout / scenario filter)
5. Polls until `complete` or `failed`

Prereqs: orchestrator running on `http://localhost:9090`, at least one worker registered.

In [ ]:
import json, time, uuid
from datetime import datetime, timezone
import requests

ORCH = "http://localhost:9090"
API  = f"{ORCH}/api/v1/benchmark"

def _unwrap(r):
    r.raise_for_status()
    body = r.json()
    return body.get("data", body) if isinstance(body, dict) else body

# ── thin client ─────────────────────────────────────────
class Bench:
    @staticmethod
    def health():
        return requests.get(f"{ORCH}/api/v1/health", timeout=2).json()
    @staticmethod
    def workers():
        return _unwrap(requests.get(f"{ORCH}/api/v1/workers", timeout=5))
    # suites
    @staticmethod
    def list_suites():
        return _unwrap(requests.get(f"{API}/suites", timeout=5))
    @staticmethod
    def get_suite(sid):
        return _unwrap(requests.get(f"{API}/suites/{sid}", timeout=5))
    @staticmethod
    def create_suite(suite):
        return _unwrap(requests.post(f"{API}/suites", json=suite, timeout=10))
    @staticmethod
    def update_suite(sid, suite):
        return _unwrap(requests.put(f"{API}/suites/{sid}", json=suite, timeout=10))
    @staticmethod
    def delete_suite(sid):
        return _unwrap(requests.delete(f"{API}/suites/{sid}", timeout=5))
    # experiments
    @staticmethod
    def list_experiments():
        return _unwrap(requests.get(f"{API}/experiments", timeout=5))
    @staticmethod
    def get_experiment(eid):
        return _unwrap(requests.get(f"{API}/experiments/{eid}", timeout=5))
    @staticmethod
    def create_experiment(name, suite_id, variants, *, description=None, repetitions=1):
        body = {
            "name": name,
            "description": description,
            "test_suite_id": suite_id,
            "variants": variants,
            "repetitions": repetitions,
        }
        return _unwrap(requests.post(f"{API}/experiments", json=body, timeout=10))
    @staticmethod
    def run_experiment(eid, *, speed=None, timeout_s=None, scenarios=None, variants=None):
        params = {}
        if speed is not None: params["speed"] = speed
        if timeout_s is not None: params["timeout_s"] = timeout_s
        if scenarios: params["scenarios"] = ",".join(scenarios)
        if variants:  params["variants"]  = ",".join(variants)
        return _unwrap(requests.post(f"{API}/experiments/{eid}/run", params=params, timeout=10))

print("client ready")

## 1 · Health check

In [ ]:
print("health  :", Bench.health())
workers = Bench.workers()
print(f"workers : {len(workers)} registered")
for w in workers:
    print(f"   - {w.get('id')}  status={w.get('status')}  pipeline={w.get('pipeline_id')}")

## 2 · Pick or build a suite

Either reuse an existing suite by id, or build one inline below.
Set `SUITE_ID = None` to use the inline builder.

In [ ]:
SUITE_ID = None  # e.g. "my-existing-suite-id" to skip the builder

for s in Bench.list_suites():
    print(f"{s['id']}  {s['name']:<40} ({len(s['scenarios'])} scenarios)")

### 2a · Inline suite builder (skip if `SUITE_ID` is set above)

Examples cover all four input kinds: `text_prompt`, `conversation`,
`barge_in`, `synthetic_caller`. Edit / trim as needed.

In [ ]:
def make_suite(name, scenarios, description=None):
    return {
        "id": str(uuid.uuid4()),
        "name": name,
        "description": description,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "scenarios": scenarios,
    }

scenarios = [
    # ── text_prompt ─────────────────────────────────────
    {
        "id": "txt-greet",
        "name": "simple greeting",
        "input": {"type": "text_prompt", "text": "Hello, how are you today?"},
        "expected": {"transcript_contains": ["hello"], "max_latency_ms": 3000},
    },
    # ── conversation ────────────────────────────────────
    {
        "id": "conv-2turn",
        "name": "two-turn followup",
        "input": {
            "type": "conversation",
            "turns": [
                {"id": "t1", "input": {"type": "text", "text": "What time is it?"}},
                {"id": "t2", "input": {"type": "text", "text": "And in Tokyo?"}},
            ],
        },
        "expected": {"turns_completed": 2, "max_latency_ms_per_turn": 4000},
    },
    # ── barge_in ────────────────────────────────────────
    {
        "id": "barge-cancel",
        "name": "interrupt long response",
        "input": {
            "type": "barge_in",
            "initial": {"id": "i1", "input": {"type": "text", "text": "Tell me a long story."}},
            "interrupt": {
                "mode":    {"kind": "inject_text", "text": "Stop, never mind."},
                "fire_on": {"kind": "playback_offset", "offset_ms": 800},
            },
            "follow_up": {"id": "f1", "input": {"type": "text", "text": "What's the weather?"}},
        },
        "expected": {"must_interrupt": True, "max_barge_in_rtt_ms": 800, "follow_up_must_complete": True},
    },
    # ── synthetic_caller ────────────────────────────────
    {
        "id": "caller-mock",
        "name": "mock caller, 3-turn",
        "input": {
            "type": "synthetic_caller",
            "config": {
                "caller":  {"kind": "mock", "script": "basic-3turn"},
                "persona": "A curious user asking about the weather.",
                "stop":    {"max_turns": 3, "max_duration_s": 60},
            },
        },
        "expected": {"min_completed_turns": 3},
    },
]

if SUITE_ID is None:
    suite = make_suite("notebook-launched suite", scenarios)
    created = Bench.create_suite(suite)
    SUITE_ID = created["id"]
    print("created suite:", SUITE_ID)
else:
    print("reusing suite:", SUITE_ID)

## 3 · Define variants and create the experiment

Each variant references a worker pipeline; tweak `variants` to match
configs your worker exposes (see `worker.toml` / `worker-bench-*.toml`).

In [ ]:
variants = [
    {
        "id": "baseline",
        "name": "baseline pipeline",
        "worker_id": workers[0]["id"] if workers else "worker-a",
        # Optional pipeline param overrides go here, e.g.:
        # "params": {"llm.temperature": 0.2}
    },
    # add more variants for A/B comparisons
]

exp = Bench.create_experiment(
    name=f"notebook-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    suite_id=SUITE_ID,
    variants=variants,
    description="Launched from run-bench.ipynb",
    repetitions=1,
)
EXPERIMENT_ID = exp["id"]
print("created experiment:", EXPERIMENT_ID)

## 4 · Kick off the run

Optional knobs:
- `speed=0.0` → as-fast-as-possible (default), `1.0` → realtime
- `timeout_s` → per-scenario cap (default 120)
- `scenarios=["id1","id2"]` → only run a subset
- `variants=["baseline"]` → only run a subset

In [ ]:
Bench.run_experiment(
    EXPERIMENT_ID,
    speed=0.0,
    timeout_s=120,
    # scenarios=["txt-greet"],
)
print("run kicked off:", EXPERIMENT_ID)

## 5 · Poll until done

Prints one progress line per poll. Stops on `complete` or `failed`.
Safe to interrupt — the orchestrator keeps running.

In [ ]:
POLL_EVERY_S   = 5
WALL_TIMEOUT_S = 30 * 60  # give up after 30 min

deadline = time.time() + WALL_TIMEOUT_S
while True:
    e = Bench.get_experiment(EXPERIMENT_ID)
    status = e.get("status")
    progress = e.get("progress") or {}
    done = progress.get("completed", "?")
    total = progress.get("total", "?")
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] status={status:<10} progress={done}/{total}")
    if status in ("complete", "failed"):
        break
    if time.time() > deadline:
        print("wall timeout reached; experiment still running on the orchestrator")
        break
    time.sleep(POLL_EVERY_S)

print("\nView results in the admin UI:")
print(f"  http://localhost:5173/benchmarks/{EXPERIMENT_ID}")